In [ ]:
# =========================================================
# 1. IMPORTS
# =========================================================
print("Loading imports...")
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore') # Suppresses Pandas/Sklearn deprecation warnings for cleaner output

from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import PredefinedSplit
from sklearn.model_selection import TimeSeriesSplit
from skopt import BayesSearchCV
from skopt.space import Real

# =========================================================
# 2. LOAD DATA
# =========================================================
print("Loading data...")
# parse_dates ensures date columns are loaded as datetime objects, saving a conversion step later
train = pd.read_csv('store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
stores = pd.read_csv('store-sales-time-series-forecasting/stores.csv')
holidays_events = pd.read_csv('store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])

# =========================================================
# 3. DATA MERGING & PREPARATION
# =========================================================
print("Merging data...")
# Merge store metadata (city, state, type, cluster) into our main sets
train = train.merge(stores, on='store_nbr', how='left')
test = test.merge(stores, on='store_nbr', how='left')

# Prepare Holidays
# In Ecuador, if a holiday falls on a weekend, it is often 'transferred' to a weekday.
# We filter out the original date of transferred holidays so we don't count them twice.
valid_holidays = holidays_events[holidays_events['transferred'] == False].drop_duplicates(subset=['date'])
valid_holidays = valid_holidays[['date', 'type']].rename(columns={'type': 'holiday_type'})

train = train.merge(valid_holidays, on='date', how='left')
test = test.merge(valid_holidays, on='date', how='left')

# Combine train and test to ensure consistent feature engineering
# This prevents errors where a specific category (e.g., a specific holiday) appears 
# in the test set but not the training set, which would break the OneHotEncoder.
train['is_test'] = False
test['is_test'] = True
test['sales'] = 0.0 # Placeholder target for the test set

df = pd.concat([train, test], ignore_index=True)

# The competition metric is Root Mean Squared Logarithmic Error (RMSLE).
# By applying log1p (log(1 + x)) to the target now, we can just use standard 
# RMSE as our loss function in the model.
df['log1p_sales'] = np.log1p(df['sales'])

# =========================================================
# 4. FEATURE ENGINEERING
# =========================================================
print("Engineering features...")

# --- Date & Fourier Features ---
# Day of week is strictly categorical
df['dayofweek'] = df['date'].dt.dayofweek.astype(str)
df['dayofyear'] = df['date'].dt.dayofyear

# Fourier terms (sin/cos) model annual seasonality effectively. 
for i in range(1, 4):
    df[f'sin_{i}'] = np.sin(2 * np.pi * i * df['dayofyear'] / 365.25)
    df[f'cos_{i}'] = np.cos(2 * np.pi * i * df['dayofyear'] / 365.25)

# --- Earthquake Feature ---
# The devastating earthquake occurred on April 16, 2016. It drastically affected 
# supermarket sales (panic buying, relief supplies) for ~4 weeks.
df['earthquake_impact'] = ((df['date'] >= '2016-04-16') & (df['date'] <= '2016-05-16')).astype(int)

# --- Payday Feature ---
# Wages in the public sector are paid every 15th and on the last day of the month.
# Sales spike during these times. This calculates days since the most recent payday.
df['day'] = df['date'].dt.day
df['is_month_end'] = df['date'].dt.is_month_end
df['days_since_payday'] = np.where(
    df['day'] < 15, 
    df['day'],  # Logic for the 1st through the 14th
    np.where(
        df['day'] == 15,
        0,          # The 15th is payday
        np.where(
            df['is_month_end'],
            0,          # The last day of the month is payday
            df['day'] - 15  # Logic for the 16th up to the day before month-end
        )
    )
)
df = df.drop(columns=['day', 'is_month_end'])

# --- Holiday Features ---
# Convert holiday presence into a simple binary feature
df['is_holiday'] = df['holiday_type'].notnull().astype(int)

# --- Interaction Terms ---
# Combining features can help linear models capture non-linear relationships.
# E.g., A promotion on 'BEVERAGES' might behave very differently than a promotion on 'BOOKS'.
df['family_promo_interaction'] = df['family'].astype(str) + "_promo_" + df['onpromotion'].astype(str)

df['family_cluster_interaction'] = df['family'].astype(str) + "_promo_" + df['cluster'].astype(str)

# --- Global Lag Features (16, 21, and 28 Days) ---
# Group by store and family, then shift the target variable backwards.
# The Kaggle test set is 15 days long, so we use 16+ day lags to avoid 
# the need for recursive predictions.
# 16 days is used as it is the soonest available.
# 21 and 28 days represent exactly 3 and 4 weeks prior, aligning with day-of-week shopping habits.
lag_days = [16, 21, 28]
for lag in lag_days:
    df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['log1p_sales'].shift(lag)

# Shifting creates NaNs at the start of the timeline. We fill with 0 temporarily.
# We will drop the first 28 days of the training set later to prevent training on this artificial data.
lag_cols = [f'lag_{lag}' for lag in lag_days]
df[lag_cols] = df[lag_cols].fillna(0)


# =========================================================
# 5. SETUP THE PIPELINE
# =========================================================
# Explicitly define feature types so the ColumnTransformer knows how to handle them
categorical_cols = ['store_nbr', 'family', 'city', 'state', 'dayofweek', 'family_promo_interaction', 'family_cluster_interaction']
numeric_cols = [
    'onpromotion', 'sin_1', 'cos_1', 'sin_2', 'cos_2', 'sin_3', 'cos_3', 
    'earthquake_impact', 'days_since_payday', 'is_holiday'
] + lag_cols

preprocessor = ColumnTransformer(
    transformers=[
        # sparse_output=True is memory efficient. handle_unknown='ignore' prevents 
        # crashes if a new category unexpectedly appears during inference.
        # We do not use the drop parameter of OneHotEncoder since we will use Ridge which will handle the multi-collinearity.
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # Ridge regression prevents overfitting by penalizing large coefficients.
    # We use solver='sparse_cg' (Conjugate Gradient) because it is highly optimized 
    # for the sparse matrices generated by our OneHotEncoder.
    ('regressor', Ridge(alpha=1.0, solver='sparse_cg')) 
])

# =========================================================
# 6. TRAIN / VALIDATION SPLIT
# =========================================================
print("Splitting data...")
train_full = df[~df['is_test']].copy()
test_full = df[df['is_test']].copy()

# Filter out the first 28 days of training data since our lag_28 feature is entirely 
# NaNs (filled with 0s) during this period. Training on it would confuse the model.
train_full = train_full[train_full['date'] >= train_full['date'].min() + pd.Timedelta(days=28)]

# Standard temporal split: use the last 15 days of August 2017 as validation, 
# mimicking the public test set timeframe.
val_start_date = '2017-08-01'

X_train = train_full[train_full['date'] < val_start_date]
y_train = X_train['log1p_sales']

X_val = train_full[train_full['date'] >= val_start_date]
y_val = X_val['log1p_sales']

# Combine them for the SearchCV
X_search = pd.concat([X_train, X_val])
y_search = pd.concat([y_train, y_val])

# Create the PredefinedSplit array:
# -1 indicates a row belongs to the training set.
#  0 indicates a row belongs to the validation set.
test_fold = np.concatenate([
    np.full(len(X_train), -1),
    np.full(len(X_val), 0)
])
ps = PredefinedSplit(test_fold)

# =========================================================
# 7. HYPERPARAMETER TUNING (BAYES SEARCH)
# =========================================================
print("Setting up BayesSearchCV...")
search_space = {
    # Search for the optimal L2 regularization strength (alpha) on a logarithmic scale
    'regressor__alpha': Real(1e-3, 1e6, prior='log-uniform')
}

bayes_search_OOS = BayesSearchCV(
    estimator=model_pipeline,
    search_spaces=search_space,
    n_iter=50,                              
    cv=ps,   # Use the PredefinedSplit here                             
    scoring='neg_root_mean_squared_error',  
    n_jobs=-1, # Utilize all available CPU cores                             
    random_state=0,
    verbose=0
)

# Crucial for Time Series: Standard K-Fold would leak future data into the past. 
# TimeSeriesSplit ensures we only ever train on the past to predict the future.
tscv = TimeSeriesSplit(n_splits=5)

bayes_search_CV = BayesSearchCV(
    estimator=model_pipeline,
    search_spaces=search_space,
    n_iter=50,                              
    cv=tscv,                                
    scoring='neg_root_mean_squared_error',  
    n_jobs=-1, # Utilize all available CPU cores                             
    random_state=0,
    verbose=0
)

print("Running Bayesian Optimization (OOS) to find best alpha...")
# Fit on the combined search set; 'ps' ensures it only trains on X_train and scores on X_val
bayes_search_OOS.fit(X_search, y_search)

print("Running Bayesian Optimization (CV) to find best alpha...")
bayes_search_CV.fit(X_train, y_train)

best_OOS_alpha = bayes_search_OOS.best_params_['regressor__alpha']
print(f"\nBest (OOS) Alpha found: {best_OOS_alpha:.4f}")
print(f"Best (OOS) Validation RMSE: {-bayes_search_OOS.best_score_:.4f}")

best_CV_alpha = bayes_search_CV.best_params_['regressor__alpha']
print(f"\nBest (CV) Alpha found: {best_CV_alpha:.4f}")
print(f"Best (CV) Validation RMSE: {-bayes_search_CV.best_score_:.4f}")

# Extract the tuned model for final training
best_OOS_model_pipeline = bayes_search_OOS.best_estimator_
best_CV_model_pipeline = bayes_search_CV.best_estimator_

# =========================================================
# 8. VALIDATION EVALUATION
# =========================================================
print("\nPredicting on validation set with tuned model...")
val_preds_OOS = best_OOS_model_pipeline.predict(X_val)
val_preds_CV = best_CV_model_pipeline.predict(X_val)

# ReLU clipping: Linear models can sometimes predict negative sales.
# We cap the minimum prediction at 0, as you cannot sell negative items.
val_preds_OOS_relu = np.maximum(0, val_preds_OOS)
val_preds_CV_relu = np.maximum(0, val_preds_CV)

val_rmsle_OOS = root_mean_squared_error(y_val, val_preds_OOS_relu)
print(f"Out-of-Sample RMSLE (Tuned Ridge): {val_rmsle_OOS:.4f}")

val_rmsle_CV = root_mean_squared_error(y_val, val_preds_CV_relu)
print(f"(CV) Out-of-Sample RMSLE (Tuned Ridge): {val_rmsle_CV:.4f}")

# =========================================================
# 9. FULL TRAINING & SUBMISSION
# =========================================================
print("\nRetraining on full dataset for submission...")
X_full_train = train_full.drop(columns=['log1p_sales', 'sales'])
y_full_train = train_full['log1p_sales']

# Refit the best pipeline on ALL available training data (including validation data)
# to give the final model the absolute most recent trends before predicting the test set.
best_OOS_model_pipeline.fit(X_full_train, y_full_train)
best_CV_model_pipeline.fit(X_full_train, y_full_train)

print("Predicting on test set...")
X_test = test_full.drop(columns=['log1p_sales', 'sales'])

test_preds_OOS = best_OOS_model_pipeline.predict(X_test)
test_preds_OOS_relu = np.maximum(0, test_preds_OOS)

test_preds_CV = best_CV_model_pipeline.predict(X_test)
test_preds_CV_relu = np.maximum(0, test_preds_CV)

# Format and save submission
submission_7_OOS = test_full[['id']].copy()
submission_7_CV = test_full[['id']].copy()
# Reverse the log1p transformation using expm1 (exp(x) - 1) to get actual sales figures
submission_7_OOS['sales'] = np.expm1(test_preds_OOS_relu)
submission_7_CV['sales'] = np.expm1(test_preds_CV_relu)

submission_7_OOS.to_csv('submission_7_OOS.csv', index=False)
print("Successfully generated submission_7_OOS.csv")

submission_7_CV.to_csv('submission_7_CV.csv', index=False)
print("Successfully generated submission_7_CV.csv")